# Spatial viewer for MOSTA (mouse) and HESTA (human)

Pick one section from the MOSTA or HESTA data found in `get_data_dir()`, then look at it in the spatial domain:

| Tab | Shows |
|---|---|
| Tissue map | bins coloured by tissue label (`annotation` in MOSTA, `celltype` in HESTA) |
| Gene map | expression of one gene (searchable box) |
| By tissue | distribution of that gene's expression per tissue label |
| Gene panel | up to 6 genes side by side, zoom linked |
| RGB overlay | the first 3 panel genes as red / green / blue, so co-expression shows as mixed colours |
| Top genes | genes most enriched in a tissue; pick one to jump to its gene map |

Values are the log-normalized `.X`; the toggle switches to raw counts (`layers['count']` in MOSTA, `layers['counts']` in HESTA).
Only the visible tab is drawn, so large sections stay responsive. Sections are discovered by scanning the data directory, so each machine lists the files it has.

In [1]:
import re
import sys
from math import ceil

sys.path.append("..")

import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import plotly.graph_objects as go
import ipywidgets as w
from anndata.io import read_elem
from IPython.display import clear_output, display
from plotly.subplots import make_subplots

from mgw.config import get_data_dir
from mgw.align_vis import CATEGORY_COLORS

# ----------------------------
# Parameters
# ----------------------------
DATA_DIR = get_data_dir()  # asks on first run, then remembered per machine

# per dataset: sub-folder, file pattern, name suffix, raw-count layer, tissue-label column in obs
DATASETS = {
    "MOSTA": dict(subdir="mosta", pattern="*.MOSTA.h5ad", suffix=".MOSTA",
                  counts_layer="count", label_col="annotation"),
    "HESTA": dict(subdir="hesta", pattern="*_HESTA.h5ad", suffix="_HESTA",
                  counts_layer="counts", label_col="celltype"),
}
LABEL_FALLBACKS = ("annotation", "celltype", "primaryanno", "simpleanno")

MARKER_GENES = ["SOX2", "PAX6", "TNNT2", "ALB", "MYOD1"]  # defaults on load (those present)
MAX_PANEL_GENES = 6
PANEL_COLS = 3
CLIP_PERCENTILE = 99     # RGB overlay: each channel is scaled to this percentile of the gene
TOP_N = 25               # rows in the top-genes table
MIN_PCT_IN = 0.1         # top genes: gene must be detected in >= this fraction of the tissue's bins
FLIP_Y = True            # image convention: y grows downwards
COLORSCALE = "Viridis"

## Section discovery and loading

In [2]:
def _natural(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


def discover_sections():
    # {"MOSTA E10.5_E1S1": ("MOSTA", Path), ...} for every file found under DATA_DIR
    found = {}
    for ds, cfg in DATASETS.items():
        files = [p for p in (DATA_DIR / cfg["subdir"]).glob(cfg["pattern"]) if not p.name.startswith(".")]
        for p in sorted(files, key=lambda p: _natural(p.name)):
            name = p.name.removesuffix(".h5ad").removesuffix(cfg["suffix"])
            found[f"{ds} {name}"] = (ds, p)
    return found


SECTIONS = discover_sections()
if not SECTIONS:
    raise FileNotFoundError(f"No MOSTA/HESTA .h5ad files found under {DATA_DIR}")
print(f"{len(SECTIONS)} sections under {DATA_DIR}:")
for key, (_, path) in SECTIONS.items():
    print(f"  {key:<24} {path.stat().st_size / 1e9:5.2f} GB")

13 sections under /Users/frederik/Documents/GitHub/MGW/data:
  MOSTA E9.5_E1S1           0.46 GB
  MOSTA E9.5_E2S1           0.40 GB
  MOSTA E10.5_E1S1          1.41 GB
  MOSTA E10.5_E2S1          0.40 GB
  MOSTA E11.5_E1S2          1.28 GB
  HESTA CS12-13_E1S3        0.31 GB
  HESTA CS12-13_E2S6        0.70 GB
  HESTA CS14-15_E2S5        0.59 GB
  HESTA CS14-15_E3S4        1.06 GB
  HESTA CS17_E1S8           1.46 GB
  HESTA CS17_E1S9           1.83 GB
  HESTA CS18_E1S3           1.83 GB
  HESTA CS18_E1S4           2.24 GB


In [3]:
def read_labels(h, col):
    # tissue labels as a Categorical; MOSTA files use the legacy encoding (int codes + obs/__categories/<col>)
    obs = h["obs"]
    if "__categories" in obs and col in obs["__categories"]:
        cats = [c.decode() if isinstance(c, bytes) else str(c) for c in obs["__categories"][col][:]]
        return pd.Categorical.from_codes(obs[col][:], cats)
    return pd.Categorical(read_elem(obs[col]))


class Section:
    # One section held in memory: expression as CSC (fast gene columns), coordinates and tissue labels.

    def __init__(self, key, ds, path):
        cfg = DATASETS[ds]
        self.key, self.ds, self.path, self.cfg = key, ds, path, cfg
        self._counts = None
        with h5py.File(path, "r") as h:
            self.genes = np.asarray(read_elem(h["var"]).index.astype(str))
            self.label_col = next(c for c in (cfg["label_col"], *LABEL_FALLBACKS) if c in h["obs"])
            self.labels = read_labels(h, self.label_col)
            self.xy = np.asarray(h["obsm"]["spatial"][:, :2], dtype=float)  # HESTA stores uint32
            self.X = sp.csc_matrix(read_elem(h["X"]))
            colors_key = f"{self.label_col}_colors"
            stored = list(read_elem(h["uns"][colors_key])) if colors_key in h["uns"] else []
        self.n = self.X.shape[0]
        self.categories = list(self.labels.categories)
        if len(stored) >= len(self.categories):
            self.colors = [str(c)[:7] for c in stored[: len(self.categories)]]  # drop the alpha digits
        else:
            self.colors = [CATEGORY_COLORS[i % len(CATEGORY_COLORS)] for i in range(len(self.categories))]
        self._index = {}
        for j, g in enumerate(self.genes):
            self._index.setdefault(g.upper(), j)

    def resolve(self, gene):
        # column index of a gene (case-insensitive), or None
        return self._index.get(str(gene).strip().upper())

    def matrix(self, raw=False):
        if not raw:
            return self.X
        if self._counts is None:
            with h5py.File(self.path, "r") as h:
                layer = self.cfg["counts_layer"]
                if layer not in h["layers"]:
                    raise ValueError(f"{self.key} has no raw-count layer '{layer}'")
                self._counts = sp.csc_matrix(read_elem(h["layers"][layer]))
        return self._counts

    def values(self, gene, raw=False):
        j = self.resolve(gene)
        if j is None:
            raise ValueError(f"'{gene}' is not a gene in {self.key}")
        return self.matrix(raw)[:, j].toarray().ravel().astype(float)

    def default_genes(self, k=3):
        # marker genes present in this section; topped up with the highest-mean genes
        picked = [self.genes[self.resolve(g)] for g in MARKER_GENES if self.resolve(g) is not None]
        if len(picked) < k:
            means = np.asarray(self.X.mean(axis=0)).ravel()
            picked += [g for g in self.genes[np.argsort(-means)] if g not in picked]
        return picked[:k]

    def default_marker_size(self):
        return 5 if self.n < 10_000 else 4 if self.n < 40_000 else 3 if self.n < 100_000 else 2

## Plot helpers

In [4]:
def _map_axes(fig, **kw):
    fig.update_xaxes(visible=False, showgrid=False, **kw)
    fig.update_yaxes(visible=False, showgrid=False, scaleanchor="x", scaleratio=1,
                     autorange="reversed" if FLIP_Y else True)


def tissue_map(sec, size):
    fig = go.Figure()
    codes = sec.labels.codes
    for i, cat in enumerate(sec.categories):
        m = codes == i
        if not m.any():
            continue
        fig.add_trace(go.Scattergl(x=sec.xy[m, 0], y=sec.xy[m, 1], mode="markers", name=f"{cat} ({m.sum():,})",
                                   marker=dict(size=size, color=sec.colors[i]), legendgroup=str(cat)))
    fig.update_layout(title=f"{sec.key}: {sec.label_col} ({sec.n:,} bins)", height=680,
                      template="plotly_white", margin=dict(l=10, r=10, t=50, b=10),
                      legend=dict(itemsizing="constant"))
    _map_axes(fig)
    return fig


def _value_label(raw):
    return "raw counts" if raw else "log-normalized"


def gene_map(sec, gene, raw, size):
    v = sec.values(gene, raw)
    o = np.argsort(v, kind="stable")  # draw high expression last
    fig = go.Figure(go.Scattergl(
        x=sec.xy[o, 0], y=sec.xy[o, 1], mode="markers",
        marker=dict(size=size, color=v[o], colorscale=COLORSCALE, cmin=0, colorbar=dict(title=_value_label(raw), thickness=12)),
    ))
    frac = (v > 0).mean()
    fig.update_layout(title=f"{gene} in {sec.key} ({_value_label(raw)}; detected in {frac:.1%} of bins)", height=680,
                      template="plotly_white", margin=dict(l=10, r=10, t=50, b=10))
    _map_axes(fig)
    return fig


def tissue_violin(sec, gene, raw):
    v = sec.values(gene, raw)
    codes = sec.labels.codes
    fig = go.Figure()
    for i, cat in enumerate(sec.categories):
        m = codes == i
        if not m.any():
            continue
        fig.add_trace(go.Violin(y=v[m], name=str(cat), marker_color=sec.colors[i], box_visible=True,
                                meanline_visible=True, points=False, spanmode="hard", scalemode="width"))
    fig.update_layout(title=f"{gene} per {sec.label_col} in {sec.key}", height=560, template="plotly_white",
                      yaxis_title=_value_label(raw), showlegend=False, margin=dict(l=10, r=10, t=50, b=10))
    return fig


def gene_panel(sec, genes, raw, size, ncols=PANEL_COLS):
    n = len(genes)
    ncols = min(ncols, n)
    nrows = ceil(n / ncols)
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=list(genes),
                        horizontal_spacing=0.07, vertical_spacing=0.10)
    for i, g in enumerate(genes):
        v = sec.values(g, raw)
        o = np.argsort(v, kind="stable")
        fig.add_trace(go.Scattergl(
            x=sec.xy[o, 0], y=sec.xy[o, 1], mode="markers",
            marker=dict(size=size, color=v[o], colorscale=COLORSCALE, cmin=0),
        ), row=i // ncols + 1, col=i % ncols + 1)
    for i in range(n):
        ax = "" if i == 0 else str(i + 1)
        fig.layout[f"xaxis{ax}"].update(visible=False, showgrid=False, matches=None if i == 0 else "x")
        fig.layout[f"yaxis{ax}"].update(visible=False, showgrid=False, scaleanchor=f"x{ax}", scaleratio=1,
                                        autorange="reversed" if FLIP_Y else True, matches=None if i == 0 else "y")
        dx, dy = fig.layout[f"xaxis{ax}"].domain, fig.layout[f"yaxis{ax}"].domain
        fig.data[i].marker.colorbar = dict(thickness=8, len=dy[1] - dy[0], y=(dy[0] + dy[1]) / 2, yanchor="middle",
                                           x=dx[1] + 0.005, xanchor="left")
    fig.update_layout(title=f"{sec.key} ({_value_label(raw)})", height=max(420, 400 * nrows), showlegend=False,
                      template="plotly_white", margin=dict(l=10, r=10, t=70, b=10))
    return fig


def rgb_overlay(sec, genes, raw, size):
    genes = list(genes)[:3]
    rgb = np.zeros((sec.n, 3))
    for c, g in enumerate(genes):
        v = sec.values(g, raw)
        hi = np.percentile(v, CLIP_PERCENTILE)
        if hi <= 0:
            hi = v.max()
        if hi > 0:
            rgb[:, c] = np.clip(v / hi, 0, 1)
    o = np.argsort(rgb.sum(axis=1), kind="stable")  # bright bins drawn last
    cols = np.rint(rgb[o] * 255).astype(int)
    fig = go.Figure(go.Scattergl(
        x=sec.xy[o, 0], y=sec.xy[o, 1], mode="markers", showlegend=False,
        marker=dict(size=size, color=[f"rgb({r},{g},{b})" for r, g, b in cols]),
    ))
    for g, chan, name in zip(genes, ("rgb(255,0,0)", "rgb(0,255,0)", "rgb(0,0,255)"), ("red", "green", "blue")):
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=f"{g} ({name})",
                                 marker=dict(size=10, color=chan)))
    fig.update_layout(title=f"{sec.key}: {' / '.join(genes)} (each scaled to its {CLIP_PERCENTILE}th percentile)",
                      height=680, template="plotly_white", plot_bgcolor="black",
                      margin=dict(l=10, r=10, t=50, b=10), legend=dict(itemsizing="constant"))
    _map_axes(fig)
    return fig


def top_genes(sec, tissue, raw, n=TOP_N, min_pct=MIN_PCT_IN):
    # genes ranked by (mean in tissue - mean elsewhere), among genes detected in >= min_pct of the tissue's bins
    X = sec.matrix(raw)
    m = np.asarray(sec.labels == tissue)
    n_in, n_out = int(m.sum()), int((~m).sum())
    if n_in == 0 or n_out == 0:
        raise ValueError(f"'{tissue}' needs bins both inside and outside the tissue")
    sum_in = X.T @ m.astype(np.float64)
    sum_all = np.asarray(X.sum(axis=0)).ravel().astype(np.float64)
    flag = m[X.indices] & (X.data > 0)  # CSC: X.indices holds the row (bin) of every stored value
    c = np.concatenate([[0], np.cumsum(flag, dtype=np.int32)])
    nz_in = c[X.indptr[1:]] - c[X.indptr[:-1]]
    nz_all = np.diff(X.indptr)
    df = pd.DataFrame({
        "mean_in": sum_in / n_in,
        "mean_out": (sum_all - sum_in) / n_out,
        "pct_in": nz_in / n_in,
        "pct_out": (nz_all - nz_in) / n_out,
    }, index=sec.genes)
    df.insert(2, "delta", df["mean_in"] - df["mean_out"])
    return df[df["pct_in"] >= min_pct].sort_values("delta", ascending=False).head(n)

## Viewer

In [5]:
STATE = {"sec": None, "busy": False}  # `busy` pauses redraws while several widgets are set at once


class _pause:
    def __enter__(self):
        STATE["busy"] = True

    def __exit__(self, *exc):
        STATE["busy"] = False


section_dd = w.Dropdown(options=list(SECTIONS), description="Section", layout=w.Layout(width="320px"))
values_tg = w.ToggleButtons(options=[("log-normalized", "log"), ("raw counts", "raw")], description="Values")
gene_cb = w.Combobox(description="Gene", placeholder="type to search", ensure_option=True,
                     continuous_update=False, layout=w.Layout(width="320px"))
multi = w.TagsInput(description="Genes", allow_duplicates=False, layout=w.Layout(width="620px"))
size_sl = w.IntSlider(value=3, min=1, max=10, description="Marker", layout=w.Layout(width="260px"))
tissue_dd = w.Dropdown(description="Tissue", layout=w.Layout(width="320px"))
pick_dd = w.Dropdown(description="Open gene", layout=w.Layout(width="320px"))
status = w.HTML()
hint = w.HTML("<i>Genes: up to %d for the panel; the first three are the R, G, B channels of the overlay.</i>" % MAX_PANEL_GENES)

TAB_TITLES = ["Tissue map", "Gene map", "By tissue", "Gene panel", "RGB overlay", "Top genes"]
outputs = [w.Output() for _ in TAB_TITLES]
tabs = w.Tab(children=outputs, titles=TAB_TITLES)


def is_raw():
    return values_tg.value == "raw"


def current_gene(sec):
    j = sec.resolve(gene_cb.value)
    if j is None:
        raise ValueError("Pick a gene in the 'Gene' box.")
    return sec.genes[j]


def chosen_genes():
    genes = list(multi.value)
    if not genes:
        raise ValueError("Add at least one gene in the 'Genes' box.")
    return genes


def draw_tissue_map(sec):
    display(tissue_map(sec, size_sl.value))


def draw_gene_map(sec):
    display(gene_map(sec, current_gene(sec), is_raw(), size_sl.value))


def draw_violin(sec):
    display(tissue_violin(sec, current_gene(sec), is_raw()))


def draw_panel(sec):
    genes = chosen_genes()
    if len(genes) > MAX_PANEL_GENES:
        raise ValueError(f"Panel shows at most {MAX_PANEL_GENES} genes; remove {len(genes) - MAX_PANEL_GENES}.")
    display(gene_panel(sec, genes, is_raw(), size_sl.value))


def draw_rgb(sec):
    display(rgb_overlay(sec, chosen_genes(), is_raw(), size_sl.value))


def draw_top_genes(sec):
    df = top_genes(sec, tissue_dd.value, is_raw())
    with _pause():
        pick_dd.options = [""] + list(df.index)
        pick_dd.value = ""
    display(df.round(3))


DRAWERS = [draw_tissue_map, draw_gene_map, draw_violin, draw_panel, draw_rgb, draw_top_genes]


def refresh(*_):
    sec = STATE["sec"]
    if STATE["busy"] or sec is None:
        return
    i = tabs.selected_index
    with outputs[i]:
        clear_output(wait=True)
        try:
            DRAWERS[i](sec)
        except ValueError as e:
            print(e)


def on_section(*_):
    status.value = f"Loading {section_dd.value} …"
    ds, path = SECTIONS[section_dd.value]
    sec = Section(section_dd.value, ds, path)
    STATE["sec"] = sec
    defaults = sec.default_genes()
    with _pause():
        gene_cb.options = list(sec.genes)
        gene_cb.value = defaults[0]
        multi.allowed_tags = list(sec.genes)
        multi.value = defaults
        tissue_dd.options = sec.categories
        size_sl.value = sec.default_marker_size()
    for out in outputs:
        out.clear_output()
    status.value = f"{sec.key}: {sec.n:,} bins, {len(sec.genes):,} genes, {len(sec.categories)} x {sec.label_col}"
    refresh()


def on_pick(change):
    if STATE["busy"] or not change["new"]:
        return
    with _pause():
        gene_cb.value = change["new"]
        tabs.selected_index = 1
    refresh()


for widget in (values_tg, gene_cb, multi, size_sl, tissue_dd, tabs):
    widget.observe(refresh, names="selected_index" if widget is tabs else "value")
section_dd.observe(on_section, names="value")
pick_dd.observe(on_pick, names="value")

ui = w.VBox([
    w.HBox([section_dd, values_tg]),
    w.HBox([gene_cb, size_sl]),
    multi,
    hint,
    w.HBox([tissue_dd, pick_dd]),
    status,
    tabs,
])
display(ui)
on_section()